# Validation and comparison with the SIR equations

Select the `simulation/.venv/bin/python` kernel. Computation and presentation are separate: first generate the report, then plot it.

The grid model is stochastic and uses local contacts. Classical SIR assumes homogeneous mixing. Agreement with the ODE is therefore not a correctness test for the grid model; we show differences between the models and also check population conservation and monotonicity of S and R.

For population fractions $s=S/N$, $i=I/N$, $r=R/N$:

$$\dot{s}=-\beta si,\qquad \dot{i}=\beta si-\gamma i,\qquad \dot{r}=\gamma i.$$

One grid step equals one time unit. By default, $\beta=4p$ and $\gamma=q$: this is a first-order approximation for four contacts, not an exact conversion from probabilities to rates. Accuracy decreases for large p and q. Custom `beta` and `gamma` values affect only the ODE, not the grid rules.

The third curve uses a discrete independent-neighbor approximation:

$$a_t=s_t[1-(1-pi_t)^4],\quad b_t=qi_t,$$
$$s_{t+1}=s_t-a_t,\quad i_{t+1}=i_t+a_t-b_t,\quad r_{t+1}=r_t+b_t.$$

This approximation also ignores spatial correlations. The shaded band shows the 10th–90th percentiles of runs, not a confidence interval for the mean.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from pprint import pprint
from simulation import create_simulation, compare_sir, sir_ode

In [ ]:
model = create_simulation(
    grid_shape=(40, 40),
    initial_infected=20,
    infection_probability=0.06,
    recover_probability=0.05,
    seed=42,
)
comparison = compare_sir(model, steps=200, repeats=20, seed=123)
# Each run starts from the same grid but uses independent random draws.
# The original model and its RNG remain unchanged.

In [ ]:
figure, axes = comparison.plot()
plt.show()
pprint(comparison.summary)

## Analytical checks and interpretation

The population should remain constant: $S+I+R=N$. S must not increase and R must not decrease. These conditions are checked for all grid runs.

For the ODE, when $s_0>0$ and $\gamma>0$:

$$\log(s/s_0)+(\beta/\gamma)(r-r_0)=0.$$

This analytical invariant should be preserved by the numerical solution. A growing epidemic peaks at $s_*=\gamma/\beta$ if initially $\beta s_0>\gamma$. The corresponding infected count is calculated in the report. The peak read from the ODE curve is sampled at one-step intervals, so it may be slightly below the analytical maximum.

The final susceptible fraction satisfies:

$$s_\infty=s_0\exp[-(\beta/\gamma)(s_0+i_0-s_\infty)].$$

We solve this equation by bisection. If initially I=0, the epidemic does not start. For gamma=0, this relation is not applied and the report returns `None`.

**Time horizon:** the formula describes infinite time. If infected individuals remain at the end of the simulation, S at the last step is not yet the final susceptible count. The report gives both values under different names. RMSE describes differences between models; we do not set an arbitrary threshold for passing validation.

In [ ]:
summary = comparison.summary
assert summary["spatial_max_population_error"] == 0
assert summary["spatial_S_nonincreasing"]
assert summary["spatial_R_nondecreasing"]
print("ODE invariant error:", summary["ode_max_analytic_invariant_error"])
print("Analytical peak I:", summary["ode_analytic_peak_infected"])
print("Peak I sampled from the ODE:", summary["ode_peak_infected"])
print("S at infinity (ODE):", comparison.final_susceptible)
print("I at the end of the grid simulation (mean):", summary["spatial_mean_infected_at_end"])

## Check integration accuracy

The ODE curves are computed numerically using RK4, without clipping or renormalizing the states. Doubling the number of substeps should give a similar result. This numerical check is separate from agreement between the spatial model and the ODE.

In [ ]:
fine = sir_ode(
    comparison.ode_counts[0], beta=comparison.beta, gamma=comparison.gamma,
    steps=200, substeps=40,
)
print("Maximum difference between 20 and 40 substeps [people]:",
      np.max(np.abs(fine - comparison.ode_counts)))

Sources: [SIR — Duke University](https://sites.math.duke.edu/education/ccp/materials/diffcalc/sir/sir2.html), [SIR and epidemic final size — Aaron King's lecture](https://kingaa.github.io/short-course/parest/odes.html), [grid model and mean-field approximation](https://arxiv.org/abs/nlin/0207048).